<a href="https://colab.research.google.com/github/blancavazquez/PLN/blob/2027-01/notebooks/2_1_Word_embedidngs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Word embeddings


El objetivo de esta libreta es una introducción a la construcción de word embeddings. Se entrenará un word embeddings desde cero usando un modelo simple de Keras para el análisis de sentimientos.


Fuente: [Word embeddings](https://www.tensorflow.org/text/guide/word_embeddings)

In [ ]:
#Carga de bibliotecas
import io
import os
import re
import shutil
import string
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.layers import TextVectorization

In [ ]:
#Montando Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Descargar del dataset de IMDb
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset = tf.keras.utils.get_file("aclImdb_v1.tar.gz", url,
                                  untar=True, cache_dir='.',
                                  cache_subdir='')

In [ ]:
dataset_dir = os.path.join(os.path.dirname(dataset), '/content/aclImdb_v1_extracted/aclImdb/')
os.listdir(dataset_dir)

In [ ]:
train_dir = os.path.join(dataset_dir, 'train')
os.listdir(train_dir)

In [ ]:
remove_dir = os.path.join(train_dir, 'unsup')
shutil.rmtree(remove_dir)

In [ ]:
batch_size = 1024
seed = 123
train_ds = tf.keras.utils.text_dataset_from_directory('/content/aclImdb_v1_extracted/aclImdb/train',
                                                      batch_size=batch_size, validation_split=0.2,
                                                      subset='training', seed=seed)
val_ds = tf.keras.utils.text_dataset_from_directory('/content/aclImdb_v1_extracted/aclImdb/train',
                                                    batch_size=batch_size, validation_split=0.2,
                                                    subset='validation', seed=seed)

In [ ]:
#Revisando etiquetas
for text_batch, label_batch in train_ds.take(1):
  for i in range(5):
    print(label_batch[i].numpy(), text_batch.numpy()[i])

# Configuración del dataset

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
#Definiendo una capa de embedding
# Embed a 1,000 word vocabulary into 5 dimensions.
embedding_layer = tf.keras.layers.Embedding(1000, 5) #Cuando se crea esta capa, los pesos se inicializan aleatoriamente (como cualquier otra capa).

In [ ]:
result = embedding_layer(tf.constant([1, 2, 3]))
result.numpy()

Para problemas de texto o secuencia, la capa de embeddings utiliza un tensor 2D de enteros, de forma (samples, sequence_length), donde cada entrada es una secuencia de enteros. Puede embeberse secuencias de longitudes variables. Se pueden introducir en esta capa lotes con formas (32, 10) (lote de 32 secuencias de longitud 10) o (64, 15) (lote de 64 secuencias de longitud 15).


El tensor devuelto tiene un eje más que la entrada; los vectores de embeddings se alinean con el nuevo último eje. Si la entrada es (2, 3), la salida es (2, 3, N).

In [ ]:
result = embedding_layer(tf.constant([[0, 1, 2], [3, 4, 5]]))
result.shape

# Procesamiento de texto

In [ ]:
# Create a custom standardization function to strip HTML break tags '<br />'.
def custom_standardization(input_data):
  lowercase = tf.strings.lower(input_data)
  stripped_html = tf.strings.regex_replace(lowercase, '<br />', ' ')
  return tf.strings.regex_replace(stripped_html,'[%s]' % re.escape(string.punctuation), '')


# Tamaño del vocabulario y número de palabras en una secuencia.
vocab_size = 10000
sequence_length = 100

# Use the text vectorization layer to normalize, split, and map strings to
# integers. Note that the layer uses the custom standardization defined above.
# Set maximum_sequence length as all samples are not of the same length.
vectorize_layer = TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length)

# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
text_ds = train_ds.map(lambda x, y: x)
vectorize_layer.adapt(text_ds)

# Creación de un modelo de clasificación

In [ ]:
embedding_dim=16

model = Sequential([vectorize_layer,
                    Embedding(vocab_size, embedding_dim, name="embedding"),
                    GlobalAveragePooling1D(),
                    Dense(16, activation='relu'),
                    Dense(1)
])

# Compilando y entrenando el modelo

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="logs")

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[tensorboard_callback])

In [ ]:
#visualizando el modelo
model.summary()

#Visualizando los resultados

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

# Guardando los word embeddings generados

In [ ]:
weights = model.get_layer('embedding').get_weights()[0]
vocab = vectorize_layer.get_vocabulary()

In [ ]:
out_v = io.open('vectors.tsv', 'w', encoding='utf-8')
out_m = io.open('metadata.tsv', 'w', encoding='utf-8')

for index, word in enumerate(vocab):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
out_v.close()
out_m.close()

In [ ]:
try:
  from google.colab import files
  files.download('vectors.tsv')
  files.download('metadata.tsv')
except Exception:
  pass

# Visualizando embeddings


Dirígite a [Embedding Projector](https://projector.tensorflow.org/) y carga tus archivos generados.